In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Ensure timestamp numeric and also keep a parsed datetime (UTC) as required by rules
for df in (metric, trace, log, error_logs):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['ts_dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True)

# Helper to produce grouped summary for files with KPI column name
def grouped_kpi_summary(df, kpi_col):
    # rename KPI col to a common name for grouping operations
    tmp = df.rename(columns={kpi_col: 'kpi_name'}).copy()
    # Compute group aggregations: count, global P95 of value (quantile on group), min timestamp, max timestamp
    grouped = (
        tmp.groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            count=('value', 'count'),
            p95_value=('value', lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan),
            min_ts=('timestamp', 'min'),
            max_ts=('timestamp', 'max'),
        )
        .sort_values('count', ascending=False)
        .reset_index(drop=True)
    )
    return grouped

# Produce grouped summaries limited to top 20
metric_grouped = grouped_kpi_summary(metric, 'kpi_name')
metric_top20 = metric_grouped.head(20)

trace_grouped = grouped_kpi_summary(trace, 'trace_name')
trace_top20 = trace_grouped.head(20)

log_grouped = grouped_kpi_summary(log, 'log_name')
log_top20 = log_grouped.head(20)

# Overall min/max timestamps per file (keep numeric original values)
files_minmax = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv', 'error_logs.csv'],
    'min_timestamp': [
        int(metric['timestamp'].min()) if not metric['timestamp'].isna().all() else pd.NA,
        int(trace['timestamp'].min()) if not trace['timestamp'].isna().all() else pd.NA,
        int(log['timestamp'].min()) if not log['timestamp'].isna().all() else pd.NA,
        int(error_logs['timestamp'].min()) if not error_logs['timestamp'].isna().all() else pd.NA,
    ],
    'max_timestamp': [
        int(metric['timestamp'].max()) if not metric['timestamp'].isna().all() else pd.NA,
        int(trace['timestamp'].max()) if not trace['timestamp'].isna().all() else pd.NA,
        int(log['timestamp'].max()) if not log['timestamp'].isna().all() else pd.NA,
        int(error_logs['timestamp'].max()) if not error_logs['timestamp'].isna().all() else pd.NA,
    ]
})

# Process error_logs: counts per cmdb_id, earliest timestamp per cmdb_id, up to 3 sample messages (earliest rows)
err = error_logs.copy()
# Ensure message column exists
if 'message' not in err.columns:
    err['message'] = pd.NA

# Compute counts and earliest timestamp per cmdb_id
err_counts = (
    err.groupby('cmdb_id', as_index=False)
    .agg(
        count=('message', 'count'),
        earliest_ts=('timestamp', 'min')
    )
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

# Get up to 3 earliest messages per cmdb_id
err_sorted = err.sort_values(['cmdb_id', 'timestamp'], ascending=[True, True])
err_earliest_samples = err_sorted.groupby('cmdb_id').head(3).copy()

# Aggregate sample messages per cmdb_id into a single string (one per earliest rows, up to 3)
sample_msgs = (
    err_earliest_samples.groupby('cmdb_id', as_index=False)
    .agg(sample_messages=('message', lambda msgs: " || ".join(map(str, msgs.tolist()))),
         sample_count=('message', 'count'))
)

# Join counts with sample messages
err_summary = (
    err_counts.merge(sample_msgs, on='cmdb_id', how='left')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

# Limit to top 10 cmdb_ids by count
err_top10 = err_summary.head(10)

# Keep final displayed outputs compact as requested:
# metric_top20, trace_top20, log_top20, err_top10, files_minmax

metric_top20, trace_top20, log_top20, err_top10, files_minmax

```
Out[1]:
```
summary = (
    "Summary of telemetry files (compact):\n\n"
    "1) Time ranges:\n"
    "- metric.csv: min=1705673040, max=1705674480\n"
    "- trace.csv:  min=1705673040, max=1705674480\n"
    "- log.csv:    min=1705673040, max=1705674480\n"
    "- error_logs.csv: no records (empty), no timestamps present\n\n"
    "2) metric.csv (top groups by row count, up to 20 shown):\n"
    "- Many KPI groups have 25 samples. Top components seen: adservice, cartservice, checkoutservice.\n"
    "- Example high-level groups: adservice (cpu, diskio, latency-50, latency-90, mem, socket, workload),\n"
    "  cartservice (cpu, diskio, latency-50, latency-90, mem, socket, workload),\n"
    "  checkoutservice (cpu, latency-50/90, mem, socket, workload).\n"
    "- P95 values vary by KPI (e.g., adservice mem p95 ~1.08e8, cartservice mem p95 ~4.48e7). All these groups span the full metric time range above.\n\n"
    "3) trace.csv (top groups by row count):\n"
    "- Dominated by checkoutservice trace metrics (25 samples each).\n"
    "- Notable trace KPIs: trace.from_frontendservice.duration_mean (~2.256s) and duration_p95 (~7.045s),\n"
    "  trace.from_checkoutservice and trace.from_root series also present. All trace groups cover the full trace time range above.\n\n"
    "4) log.csv (top groups by row count):\n"
    "- log.row_count and log.error_count groups appear for many services (typically 25 samples).\n"
    "- Highest p95 log.row_count values: frontend (~2776), currencyservice (~2489), recommendationservice (~590.2), shippingservice (~504.4).\n"
    "- log.error_count is 0.0 for the top groups shown. emailservice and paymentservice have 24 samples; redis only 5 samples.\n\n"
    "5) error_logs.csv:\n"
    "- No error log records were found (empty summary). No per-cmdb_id samples to report.\n\n"
    "Overall: metric/trace/log telemetry are present and aligned over the same minute range (1705673040–1705674480). "
    "Trace data is concentrated on checkoutservice, and logs show high row_count (traffic) for frontend and currencyservice. "
    "There are no error log entries in error_logs.csv. Next steps: if you want incident-root analysis, filter these series to the incident window and compute deltas/anomalies."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  count     p95_value      min_ts      max_ts
0         adservice         cpu     25  7.858440e-01  1705673040  1705674480
1         adservice      diskio     25  7.043340e+04  1705673040  1705674480
2         adservice  latency-50     25  3.016818e-03  1705673040  1705674480
3         adservice  latency-90     25  4.635089e-03  1705673040  1705674480
4         adservice         mem     25  1.084068e+08  1705673040  1705674480
5         adservice      socket     25  4.000000e+00  1705673040  1705674480
6         adservice    workload     25  1.540453e+01  1705673040  1705674480
7       cartservice         cpu     25  2.674623e+00  1705673040  1705674480
8       cartservice      diskio     25  0.000000e+00  1705673040  1705674480
9       cartservice  latency-50     25  3.220329e-03  1705673040  1705674480
10      cartservice  latency-90     25  5.614494e-03  1705673040  1705674480
11      cartservice         mem     25  4.480035e+07  1705673040  1705674480
12      cartservice      socket     25  4.000000e+00  1705673040  1705674480
13      cartservice    workload     25  2.595706e+01  1705673040  1705674480
14  checkoutservice         cpu     25  4.751291e-01  1705673040  1705674480
15  checkoutservice  latency-50     25  1.905930e+00  1705673040  1705674480
16  checkoutservice  latency-90     25  4.840979e+00  1705673040  1705674480
17  checkoutservice         mem     25  1.230123e+07  1705673040  1705674480
18  checkoutservice      socket     25  9.000000e+00  1705673040  1705674480
19  checkoutservice    workload     25  1.008370e+00  1705673040  1705674480,             cmdb_id                                  kpi_name  count   p95_value      min_ts      max_ts
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.320425  1705673040  1705674480
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    1.188384  1705673040  1705674480
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000  1705673040  1705674480
3   checkoutservice      trace.from_checkoutservice.row_count     25  295.000000  1705673040  1705674480
4   checkoutservice  trace.from_frontendservice.duration_mean     25    2.256292  1705673040  1705674480
5   checkoutservice   trace.from_frontendservice.duration_p95     25    7.045089  1705673040  1705674480
6   checkoutservice     trace.from_frontendservice.error_rate     25    0.000000  1705673040  1705674480
7   checkoutservice      trace.from_frontendservice.row_count     25   34.800000  1705673040  1705674480
8   checkoutservice             trace.from_root.duration_mean     25    0.224216  1705673040  1705674480
9   checkoutservice              trace.from_root.duration_p95     25    0.919963  1705673040  1705674480
10  checkoutservice                trace.from_root.error_rate     25    0.000000  1705673040  1705674480
11  checkoutservice                 trace.from_root.row_count     25  150.000000  1705673040  1705674480
12  checkoutservice    trace.to_checkoutservice.duration_mean     25    0.320425  1705673040  1705674480
13  checkoutservice     trace.to_checkoutservice.duration_p95     25    1.188384  1705673040  1705674480
14  checkoutservice       trace.to_checkoutservice.error_rate     25    0.000000  1705673040  1705674480
15  checkoutservice        trace.to_checkoutservice.row_count     25  295.000000  1705673040  1705674480
16  checkoutservice    trace.to_currencyservice.duration_mean     25    0.001619  1705673040  1705674480
17  checkoutservice     trace.to_currencyservice.duration_p95     25    0.000324  1705673040  1705674480
18  checkoutservice       trace.to_currencyservice.error_rate     25    0.000000  1705673040  1705674480
19  checkoutservice        trace.to_currencyservice.row_count     25  127.600000  1705673040  1705674480,                   cmdb_id         kpi_name  count  p95_value      min_ts      max_ts
0               adservice  log.error_count     25        0.0  1705673040  1705674480
1               adservice    log.row_count     25      461.4  1705673040  1705674480
2             cartservice  log.error_count     25        0.0  1705673040  1705674480
3             cartservice    log.row_count     25      810.2  1705673040  1705674480
4         checkoutservice  log.error_count     25        0.0  1705673040  1705674480
5         checkoutservice    log.row_count     25      100.6  1705673040  1705674480
6         currencyservice  log.error_count     25        0.0  1705673040  1705674480
7         currencyservice    log.row_count     25     2489.0  1705673040  1705674480
8                frontend    log.row_count     25     2776.0  1705673040  1705674480
9                frontend  log.error_count     25        0.0  1705673040  1705674480
10  recommendationservice  log.error_count     25        0.0  1705673040  1705674480
11  recommendationservice    log.row_count     25      590.2  1705673040  1705674480
12        shippingservice  log.error_count     25        0.0  1705673040  1705674480
13        shippingservice    log.row_count     25      504.4  1705673040  1705674480
14           emailservice  log.error_count     24        0.0  1705673040  1705674420
15           emailservice    log.row_count     24       33.0  1705673040  1705674420
16         paymentservice  log.error_count     24        0.0  1705673040  1705674420
17         paymentservice    log.row_count     24       66.0  1705673040  1705674420
18                  redis    log.row_count      5        5.0  1705673220  1705674420
19                  redis  log.error_count      5        0.0  1705673220  1705674420, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, sample_messages, sample_count]
Index: [],              file min_timestamp max_timestamp
0      metric.csv    1705673040    1705674480
1       trace.csv    1705673040    1705674480
2         log.csv    1705673040    1705674480
3  error_logs.csv          <NA>          <NA>)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames: metric, trace, log (assumed loaded in kernel)
# Ensure required datetime column exists
for df in (metric, trace, log):
    if 'ts_dt' not in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True)

# Services to restrict to
services = {'checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice'}

# Incident window (UTC)
win_start = pd.Timestamp("2024-01-19 14:01:02", tz="UTC")
win_end = pd.Timestamp("2024-01-19 14:31:02", tz="UTC")

def compute_summary(df, kpi_col):
    # Restrict to selected services
    df_sel = df[df['cmdb_id'].isin(services)].copy()
    # Ensure value numeric
    df_sel['value'] = pd.to_numeric(df_sel['value'], errors='coerce')
    # Compute full-series thresholds per group (global p95 and p05) and full count
    group_thresh = (
        df_sel.groupby(['cmdb_id', kpi_col], as_index=False)
        .agg(
            full_count=('value', 'count'),
            p95=('value', lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan),
            p05=('value', lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan),
        )
    )
    # Prepare in-window rows by merging thresholds to allow comparisons
    df_in = df_sel.merge(group_thresh, on=['cmdb_id', kpi_col], how='left')
    df_in_window = df_in[(df_in['ts_dt'] >= win_start) & (df_in['ts_dt'] <= win_end)].copy()
    if not df_in_window.empty:
        df_in_window['gt_p95'] = df_in_window['value'] > df_in_window['p95']
        df_in_window['lt_p05'] = df_in_window['value'] < df_in_window['p05']
    else:
        # create empty columns for consistent aggregation
        df_in_window['gt_p95'] = pd.Series(dtype=bool)
        df_in_window['lt_p05'] = pd.Series(dtype=bool)
    # Aggregate in-window basic stats per group
    in_basic = (
        df_in_window.groupby(['cmdb_id', kpi_col], as_index=False)
        .agg(
            in_window_count=('value', 'count'),
            in_window_max=('value', 'max'),
            in_window_min=('value', 'min'),
        )
    )
    # Aggregate gt_p95/lt_p05 counts
    gt_counts = (
        df_in_window[df_in_window['gt_p95']]
        .groupby(['cmdb_id', kpi_col], as_index=False)
        .agg(in_window_gt_p95_count=('gt_p95', 'sum'),
             earliest_in_window_gt_p95_ts=('timestamp', 'min'))
    )
    lt_counts = (
        df_in_window[df_in_window['lt_p05']]
        .groupby(['cmdb_id', kpi_col], as_index=False)
        .agg(in_window_lt_p05_count=('lt_p05', 'sum'),
             earliest_in_window_lt_p05_ts=('timestamp', 'min'))
    )
    # Combine all aggregates with group_thresh to ensure groups with zero in-window rows are included
    summary = group_thresh.merge(in_basic, on=['cmdb_id', kpi_col], how='left') \
                          .merge(gt_counts, on=['cmdb_id', kpi_col], how='left') \
                          .merge(lt_counts, on=['cmdb_id', kpi_col], how='left')
    # Fill missing counts with 0 and keep earliest timestamps as NA where missing
    summary['in_window_count'] = summary['in_window_count'].fillna(0).astype(int)
    summary['in_window_gt_p95_count'] = summary['in_window_gt_p95_count'].fillna(0).astype(int)
    summary['in_window_lt_p05_count'] = summary['in_window_lt_p05_count'].fillna(0).astype(int)
    # For earliest timestamps keep numeric or pd.NA
    summary['earliest_in_window_gt_p95_ts'] = summary['earliest_in_window_gt_p95_ts'].where(
        summary['earliest_in_window_gt_p95_ts'].notna(), pd.NA
    )
    summary['earliest_in_window_lt_p05_ts'] = summary['earliest_in_window_lt_p05_ts'].where(
        summary['earliest_in_window_lt_p05_ts'].notna(), pd.NA
    )
    # For in_window_max/min keep as numeric or NA
    summary['in_window_max'] = summary['in_window_max'].where(summary['in_window_max'].notna(), pd.NA)
    summary['in_window_min'] = summary['in_window_min'].where(summary['in_window_min'].notna(), pd.NA)
    # Rename KPI col to a uniform name for display
    summary = summary.rename(columns={kpi_col: 'kpi_name',
                                      'full_count': 'full_count',
                                      'p95': 'p95',
                                      'p05': 'p05'})
    # Select and order required columns
    cols = ['cmdb_id', 'kpi_name', 'full_count', 'p95', 'p05',
            'in_window_count', 'in_window_gt_p95_count', 'earliest_in_window_gt_p95_ts',
            'in_window_lt_p05_count', 'earliest_in_window_lt_p05_ts',
            'in_window_max', 'in_window_min']
    summary = summary[cols]
    # Sort by full_count desc and limit to top 20
    summary = summary.sort_values('full_count', ascending=False).reset_index(drop=True).head(20)
    return summary

# Compute summaries for metric, trace, log
metric_summary = compute_summary(metric, 'kpi_name')
trace_summary = compute_summary(trace, 'trace_name')
log_summary = compute_summary(log, 'log_name')

# Display the three compact tables
metric_summary, trace_summary, log_summary

```
Out[2]:
```
Summary (incident window = 2024-01-19 14:01:02 → 14:31:02 UTC). I restricted analysis to these services: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice.

General note
- For virtually all groups returned, full_count = in_window_count (mostly 25, emailservice 24). That means the incident window covers the available data for these groups.
- I computed global thresholds (P95 and P05) using the entire series per group, then counted in-window points above P95 and below P05, and reported earliest timestamps (numeric Unix seconds) for those exceedances.

Highlights by file (compact):

1) metric.csv (top groups)
- checkoutservice.cpu — full_count 25, p95 0.4751291, p05 0.3017594.
  - in_window_count 25, in_window_gt_p95_count 2 (earliest >P95 at 1705673220), in_window_lt_p05_count 2 (earliest <P05 at 1705673880). in_window_max 0.4994899, in_window_min 0.2895111.
- checkoutservice.latency-50 & latency-90 & mem & workload — each show 25 points, typically 2 points >P95 and 2 <P05 with earliest timestamps reported (see above for cpu pattern).
- currencyservice.cpu — full_count 25, p95 34.14572, p05 30.77368.
  - in_window_count 25, 2 points >P95 (earliest 1705673220), 2 points <P05 (earliest 1705674240), in_window_max 34.46769, in_window_min 27.26641.
- currencyservice.mem, latency, workload — similar pattern: 25 points, usually 2 >P95 and 2 <P05, with earliest exceedance timestamps in the 1705673xxx–1705674xxx range.
- Several metric groups (e.g., socket, diskio where p95==p05==constant) had zero >P95 and zero <P05 in-window (no anomalies by these thresholds).

Overall metric observation: checkoutservice and currencyservice show repeated small numbers of points (commonly 2) exceeding global P95 and also a few points below global P05 during the incident window across CPU, latency, memory, and workload KPIs.

2) trace.csv (top groups)
- All top trace groups are for checkoutservice (25 points each).
- Example: trace.from_frontendservice.duration_p95 — full_count 25, p95 7.045089s, p05 0.190962s.
  - in_window_count 25, in_window_gt_p95_count 2 (earliest 1705673760), in_window_lt_p05_count 2 (earliest 1705673280), in_window_max 9.877074, in_window_min 0.182118.
- trace.from_checkoutservice.duration_mean and many other checkoutservice trace metrics show the same pattern: 25 points, typically 2 >P95 and 2 <P05, with earliest exceedances recorded.
- trace row_count series (e.g., trace.from_checkoutservice.row_count) show in-window maxima that indicate spikes (example max 330, min 4).

Overall trace observation: checkoutservice traces exhibit both high and low excursions relative to global thresholds during the window (several KPIs with 2 points above P95 and 2 below P05).

3) log.csv (top groups)
- checkoutservice.log.row_count — full_count 25, p95 100.6, p05 52.8.
  - in_window_count 25, in_window_gt_p95_count 2 (earliest 1705673160), in_window_lt_p05_count 2 (earliest 1705673820), in_window_max 114, in_window_min 1.
- currencyservice.log.row_count — full_count 25, p95 2489.0, p05 2246.6.
  - in_window_count 25, in_window_gt_p95_count 1 (earliest 1705673160), in_window_lt_p05_count 2 (earliest 1705674180), in_window_max 2501, in_window_min 108.
- recommendationservice.log.row_count — full_count 25, p95 590.2, p05 528.0.
  - in_window_count 25, in_window_gt_p95_count 2 (earliest 1705673160), in_window_lt_p05_count 2 (earliest 1705673100), in_window_max 604, in_window_min 19.
- emailservice.log.row_count — full_count 24, p95 33.0, p05 20.15.
  - in_window_count 24, in_window_gt_p95_count 1 (earliest 1705673160), in_window_lt_p05_count 2 (earliest 1705673100), in_window_max 38, in_window_min 17.
- log.error_count groups for these services are 0.0 (p95=0) and show no >P95 / <P05 anomalies.

Overall log observation: log.row_count shows spikes above P95 for some services (notably currencyservice, recommendationservice, checkoutservice) with earliest spike timestamps around 1705673160; emailservice has one >P95. error_count remains zero.

Compact conclusion / next steps
- The incident window contains the whole available series for these services. Multiple services (checkoutservice and currencyservice particularly) show repeated excursions above P95 and below P05 across metrics, traces, and logs — commonly 2 points >P95 and 2 points <P05 per KPI.
- Trace anomalies are concentrated in checkoutservice; log row_count spikes are evident for currencyservice, recommendationservice, and checkoutservice.
- Next steps to triage: inspect the specific timestamps around the earliest exceedances (e.g., 1705673160, 1705673220, 1705673760, 1705673880, 1705674300) and correlate metric/trace/log spikes across services to identify causal flow.

If you want, I can produce a short table of the groups that had any in-window >P95 occurrences (with their earliest >P95 timestamp) to prioritize investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  full_count           p95           p05  in_window_count  in_window_gt_p95_count  earliest_in_window_gt_p95_ts  in_window_lt_p05_count  earliest_in_window_lt_p05_ts  in_window_max  in_window_min
0   checkoutservice         cpu          25  4.751291e-01  3.017594e-01               25                       2                    1705673220                       2                    1705673880   4.994899e-01   2.895111e-01
1   checkoutservice  latency-50          25  1.905930e+00  9.788314e-02               25                       2                    1705674180                       2                    1705673280   2.068698e+00   8.456669e-02
2   checkoutservice  latency-90          25  4.840979e+00  2.216395e-01               25                       2                    1705674240                       2                    1705673280   8.493333e+00   2.215132e-01
3   checkoutservice         mem          25  1.230123e+07  1.194749e+07               25                       2                    1705673880                       2                    1705673160   1.230718e+07   1.193185e+07
4   checkoutservice      socket          25  9.000000e+00  9.000000e+00               25                       0                          <NA>                       0                          <NA>   9.000000e+00   9.000000e+00
5   checkoutservice    workload          25  1.008370e+00  3.492667e-01               25                       2                    1705673160                       2                    1705673820   1.252083e+00   3.332500e-01
6   currencyservice         cpu          25  3.414572e+01  3.077368e+01               25                       2                    1705673220                       2                    1705674240   3.446769e+01   2.726641e+01
7   currencyservice      diskio          25  0.000000e+00  0.000000e+00               25                       0                          <NA>                       0                          <NA>   0.000000e+00   0.000000e+00
8   currencyservice  latency-50          25  4.290613e-03  3.861156e-03               25                       2                    1705673400                       2                    1705673820   4.330743e-03   3.839986e-03
9   currencyservice  latency-90          25  5.729414e-02  3.897417e-02               25                       2                    1705673400                       2                    1705673820   5.933463e-02   3.811502e-02
10  currencyservice         mem          25  4.549032e+07  4.459493e+07               25                       2                    1705673160                       2                    1705673640   4.569347e+07   4.458810e+07
11  currencyservice      socket          25  3.000000e+00  3.000000e+00               25                       0                          <NA>                       0                          <NA>   3.000000e+00   3.000000e+00
12  currencyservice    workload          25  8.310254e+01  7.430881e+01               25                       2                    1705673040                       2                    1705673820   8.492290e+01   7.411223e+01
13     emailservice         cpu          25  3.350630e-01  2.710134e-01               25                       2                    1705673580                       2                    1705673820   3.416196e-01   2.558554e-01
14     emailservice      diskio          25  0.000000e+00  0.000000e+00               25                       0                          <NA>                       0                          <NA>   0.000000e+00   0.000000e+00
15     emailservice  latency-50          25  3.207629e-03  3.000000e-03               25                       2                    1705673880                       0                          <NA>   3.262016e-03   3.000000e-03
16     emailservice  latency-90          25  2.664632e-02  4.600000e-03               25                       2                    1705673820                       0                          <NA>   3.587361e-02   4.600000e-03
17     emailservice         mem          25  4.362064e+07  4.351354e+07               25                       2                    1705674060                       2                    1705673160   4.364459e+07   4.351311e+07
18     emailservice      socket          25  3.000000e+00  3.000000e+00               25                       0                          <NA>                       0                          <NA>   3.000000e+00   3.000000e+00
19     emailservice    workload          25  1.015303e+00  3.426000e-01               25                       2                    1705673160                       2                    1705674420   1.233483e+00   2.000000e-01,             cmdb_id                                  kpi_name  full_count         p95         p05  in_window_count  in_window_gt_p95_count  earliest_in_window_gt_p95_ts  in_window_lt_p05_count  earliest_in_window_lt_p05_ts  in_window_max  in_window_min
0   checkoutservice  trace.from_checkoutservice.duration_mean          25    0.320425    0.003614               25                       2                    1705674300                       2                    1705673280       0.378993       0.003451
1   checkoutservice   trace.from_checkoutservice.duration_p95          25    1.188384    0.005481               25                       2                    1705674300                       2                    1705673280       1.455685       0.005076
2   checkoutservice     trace.from_checkoutservice.error_rate          25    0.000000    0.000000               25                       0                          <NA>                       0                          <NA>       0.000000       0.000000
3   checkoutservice      trace.from_checkoutservice.row_count          25  295.000000  148.800000               25                       2                    1705673160                       2                    1705673820     330.000000       4.000000
4   checkoutservice  trace.from_frontendservice.duration_mean          25    2.256292    0.091929               25                       2                    1705673760                       2                    1705673280       3.457607       0.076753
5   checkoutservice   trace.from_frontendservice.duration_p95          25    7.045089    0.190962               25                       2                    1705673760                       2                    1705673280       9.877074       0.182118
6   checkoutservice     trace.from_frontendservice.error_rate          25    0.000000    0.000000               25                       0                          <NA>                       0                          <NA>       0.000000       0.000000
7   checkoutservice      trace.from_frontendservice.row_count          25   34.800000   17.600000               25                       2                    1705673160                       2                    1705673820      38.000000       1.000000
8   checkoutservice             trace.from_root.duration_mean          25    0.224216    0.013372               25                       2                    1705673760                       2                    1705673100       0.246154       0.010783
9   checkoutservice              trace.from_root.duration_p95          25    0.919963    0.059966               25                       2                    1705673760                       2                    1705673280       1.270512       0.051817
10  checkoutservice                trace.from_root.error_rate          25    0.000000    0.000000               25                       0                          <NA>                       0                          <NA>       0.000000       0.000000
11  checkoutservice                 trace.from_root.row_count          25  150.000000   80.600000               25                       2                    1705673160                       2                    1705673820     164.000000       3.000000
12  checkoutservice    trace.to_checkoutservice.duration_mean          25    0.320425    0.003614               25                       2                    1705674300                       2                    1705673280       0.378993       0.003451
13  checkoutservice     trace.to_checkoutservice.duration_p95          25    1.188384    0.005481               25                       2                    1705674300                       2                    1705673280       1.455685       0.005076
14  checkoutservice       trace.to_checkoutservice.error_rate          25    0.000000    0.000000               25                       0                          <NA>                       0                          <NA>       0.000000       0.000000
15  checkoutservice        trace.to_checkoutservice.row_count          25  295.000000  148.800000               25                       2                    1705673160                       2                    1705673820     330.000000       4.000000
16  checkoutservice    trace.to_currencyservice.duration_mean          25    0.001619    0.000130               25                       2                    1705673160                       2                    1705673940       0.002341       0.000114
17  checkoutservice     trace.to_currencyservice.duration_p95          25    0.000324    0.000187               25                       2                    1705673100                       2                    1705673940       0.000756       0.000141
18  checkoutservice       trace.to_currencyservice.error_rate          25    0.000000    0.000000               25                       0                          <NA>                       0                          <NA>       0.000000       0.000000
19  checkoutservice        trace.to_currencyservice.row_count          25  127.600000   59.800000               25                       2                    1705673160                       2                    1705673820     140.000000       2.000000,                  cmdb_id         kpi_name  full_count     p95      p05  in_window_count  in_window_gt_p95_count  earliest_in_window_gt_p95_ts  in_window_lt_p05_count  earliest_in_window_lt_p05_ts  in_window_max  in_window_min
0        checkoutservice  log.error_count          25     0.0     0.00               25                       0                          <NA>                       0                          <NA>              0              0
1        checkoutservice    log.row_count          25   100.6    52.80               25                       2                    1705673160                       2                    1705673820            114              1
2        currencyservice  log.error_count          25     0.0     0.00               25                       0                          <NA>                       0                          <NA>              0              0
3        currencyservice    log.row_count          25  2489.0  2246.60               25                       1                    1705673160                       2                    1705674180           2501            108
4  recommendationservice    log.row_count          25   590.2   528.00               25                       2                    1705673160                       2                    1705673100            604             19
5  recommendationservice  log.error_count          25     0.0     0.00               25                       0                          <NA>                       0                          <NA>              0              0
6           emailservice    log.row_count          24    33.0    20.15               24                       1                    1705673160                       2                    1705673100             38             17
7           emailservice  log.error_count          24     0.0     0.00               24                       0                          <NA>                       0                          <NA>              0              0)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse variables: metric, trace, log, services, win_start, win_end (already defined in kernel)
# If any are missing, reconstruct minimal required ones (but they should exist from previous steps).

# Helper to process a single dataframe and return per-group >P95 / <P05 summary
def group_events_from_df(df, kpi_col):
    df_sel = df[df['cmdb_id'].isin(services)].copy()
    df_sel['value'] = pd.to_numeric(df_sel['value'], errors='coerce')
    # Compute global thresholds (full series) per group
    group_thresh = (
        df_sel.groupby(['cmdb_id', kpi_col], as_index=False)
        .agg(
            full_count=('value', 'count'),
            p95=('value', lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan),
            p05=('value', lambda x: float(x.quantile(0.05)) if x.notna().any() else np.nan),
        )
        .rename(columns={kpi_col: 'kpi_name'})
    )
    # Merge thresholds back and filter to incident window
    merged = df_sel.merge(group_thresh, left_on=['cmdb_id', kpi_col], right_on=['cmdb_id', 'kpi_name'], how='left')
    merged['ts_dt'] = pd.to_datetime(merged['timestamp'].astype('int64'), unit='s', utc=True)
    in_win = merged[(merged['ts_dt'] >= win_start) & (merged['ts_dt'] <= win_end)].copy()
    if in_win.empty:
        return pd.DataFrame(columns=['cmdb_id','kpi_name','group_has_gt','earliest_gt_ts','group_max_severity',
                                     'group_has_lt','earliest_lt_ts'])
    # flags
    in_win['gt_p95'] = in_win['value'] > in_win['p95']
    in_win['lt_p05'] = in_win['value'] < in_win['p05']
    # severity guard (avoid divide-by-zero): set severity = value / p95 where p95>0 else NaN
    in_win['severity'] = np.where(in_win['p95'] > 0, in_win['value'] / in_win['p95'], np.nan)
    # per-group aggregation
    gt_agg = (
        in_win[in_win['gt_p95']].groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(group_has_gt=('gt_p95', 'any'),
             earliest_gt_ts=('timestamp','min'),
             group_max_severity=('severity','max'))
    )
    lt_agg = (
        in_win[in_win['lt_p05']].groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(group_has_lt=('lt_p05','any'),
             earliest_lt_ts=('timestamp','min'))
    )
    # Combine gt and lt per group (outer join)
    grp = pd.merge(group_thresh[['cmdb_id','kpi_name']], gt_agg, on=['cmdb_id','kpi_name'], how='left')
    grp = grp.merge(lt_agg, on=['cmdb_id','kpi_name'], how='left')
    # Fill boolean flags
    grp['group_has_gt'] = grp['group_has_gt'].fillna(False)
    grp['group_has_lt'] = grp['group_has_lt'].fillna(False)
    # Keep only relevant columns
    grp = grp[['cmdb_id','kpi_name','group_has_gt','earliest_gt_ts','group_max_severity','group_has_lt','earliest_lt_ts']]
    return grp

# Process each file
metric_groups = group_events_from_df(metric, 'kpi_name')
trace_groups = group_events_from_df(trace, 'trace_name')
log_groups = group_events_from_df(log, 'log_name')

# Combine group-level results across files (concatenate)
all_groups = pd.concat([metric_groups, trace_groups, log_groups], ignore_index=True, sort=False)

# For groups that may appear multiple times (unlikely across different files with same kpi_name),
# keep per-file occurrences distinct. We treat each (cmdb_id,kpi_name) row as a group occurrence.
# Now aggregate per service:
def agg_service(df_groups):
    # For gt_p95
    gt = (
        df_groups[df_groups['group_has_gt']]
        .groupby('cmdb_id', as_index=False)
        .agg(gt_p95_groups_count=('kpi_name', 'nunique'),
             earliest_gt_p95_ts=('earliest_gt_ts', 'min'),
             max_gt_p95_severity=('group_max_severity', 'max'))
    )
    # For lt_p05
    lt = (
        df_groups[df_groups['group_has_lt']]
        .groupby('cmdb_id', as_index=False)
        .agg(lt_p05_groups_count=('kpi_name', 'nunique'),
             earliest_lt_p05_ts=('earliest_lt_ts', 'min'))
    )
    svc = pd.DataFrame({'cmdb_id': list(services)})
    svc = svc.merge(gt, on='cmdb_id', how='left').merge(lt, on='cmdb_id', how='left')
    # Fill missing counts with 0 and set NA for timestamps/severity where missing
    svc['gt_p95_groups_count'] = svc['gt_p95_groups_count'].fillna(0).astype(int)
    svc['lt_p05_groups_count'] = svc['lt_p05_groups_count'].fillna(0).astype(int)
    svc['earliest_gt_p95_ts'] = svc['earliest_gt_p95_ts'].where(svc['earliest_gt_p95_ts'].notna(), pd.NA)
    svc['max_gt_p95_severity'] = svc['max_gt_p95_severity'].where(svc['max_gt_p95_severity'].notna(), pd.NA)
    svc['earliest_lt_p05_ts'] = svc['earliest_lt_p05_ts'].where(svc['earliest_lt_p05_ts'].notna(), pd.NA)
    # Order columns as requested
    svc = svc[['cmdb_id', 'gt_p95_groups_count', 'earliest_gt_p95_ts', 'max_gt_p95_severity',
               'lt_p05_groups_count', 'earliest_lt_p05_ts']]
    # Ensure only the five candidate services in requested order (sort by services set order)
    svc = svc.set_index('cmdb_id').loc[list(services)].reset_index()
    return svc

service_summary = agg_service(all_groups)

service_summary

```
Out[3]:
```
Summary (per service, numeric Unix timestamps):

- checkoutservice
  - gt_p95_groups_count: 30
  - earliest_gt_p95_ts: 1705673100
  - max_gt_p95_severity: 2.334239
  - lt_p05_groups_count: 30
  - earliest_lt_p05_ts: 1705673100
  - Note: most groups (30) show >P95 and <P05 events in-window; max severity ~2.33.

- currencyservice
  - gt_p95_groups_count: 14
  - earliest_gt_p95_ts: 1705673040
  - max_gt_p95_severity: 74.020366
  - lt_p05_groups_count: 14
  - earliest_lt_p05_ts: 1705673100
  - Note: fewer groups than checkoutservice, but a very large max severity (≈74× the group P95) — high-priority spike.

- emailservice
  - gt_p95_groups_count: 11
  - earliest_gt_p95_ts: 1705673040
  - max_gt_p95_severity: 1.525860
  - lt_p05_groups_count: 9
  - earliest_lt_p05_ts: 1705673100

- productcatalogservice
  - gt_p95_groups_count: 16
  - earliest_gt_p95_ts: 1705673040
  - max_gt_p95_severity: 1.240310
  - lt_p05_groups_count: 16
  - earliest_lt_p05_ts: 1705673100

- recommendationservice
  - gt_p95_groups_count: 20
  - earliest_gt_p95_ts: 1705673100
  - max_gt_p95_severity: 1.431872
  - lt_p05_groups_count: 20
  - earliest_lt_p05_ts: 1705673040

Overall takeaway
- checkoutservice has the largest number of distinct KPI groups showing >P95 events (30 groups).
- currencyservice has the single largest severity spike (severity ≈ 74), making it the highest-priority anomaly despite fewer affected groups.
- All five services show both >P95 and <P05 events; earliest anomaly timestamps are clustered around 1705673040–1705673100 and later.
- Recommended next step: prioritize investigating the currencyservice spike (max severity) and correlate timestamps (1705673040–1705673160 range) across metric/trace/log signals for causal analysis.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  gt_p95_groups_count  earliest_gt_p95_ts  max_gt_p95_severity  lt_p05_groups_count  earliest_lt_p05_ts
0  recommendationservice                   20          1705673100             1.431872                   20          1705673040
1  productcatalogservice                   16          1705673040             1.240310                   16          1705673100
2        currencyservice                   14          1705673040            74.020366                   14          1705673100
3           emailservice                   11          1705673040             1.525860                    9          1705673100
4        checkoutservice                   30          1705673100             2.334239                   30          1705673100```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse existing DataFrames and variables from kernel: metric, trace, log, win_start, win_end
# Define target services
target_services = ['checkoutservice', 'currencyservice']

# Ensure timestamp datetime column exists and value numeric
for df in (metric, trace, log):
    if 'ts_dt' not in df.columns:
        df['ts_dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

def find_earliest_gt_p95(df, kpi_col, svc_list):
    # Work on a copy and normalize KPI column name to 'kpi_name'
    df_sel = df[df['cmdb_id'].isin(svc_list)].copy()
    df_sel = df_sel.rename(columns={kpi_col: 'kpi_name'})
    # Compute global p95 per group using full series (restricted to svc_list)
    group_p95 = (
        df_sel.groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(group_p95=('value', lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan))
    )
    # Merge thresholds back
    merged = df_sel.merge(group_p95, on=['cmdb_id', 'kpi_name'], how='left')
    # Filter to incident window and rows > p95
    in_win = merged[(merged['ts_dt'] >= win_start) & (merged['ts_dt'] <= win_end)].copy()
    in_win = in_win[in_win['value'] > in_win['group_p95']]
    if in_win.empty:
        return pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'group_p95',
                                     'earliest_gt_p95_ts', 'value_at_earliest_gt_p95', 'severity'])
    # For each group pick earliest timestamp row
    idx = in_win.groupby(['cmdb_id', 'kpi_name'])['timestamp'].idxmin()
    earliest_rows = in_win.loc[idx].copy()
    # Compute severity, guard p95==0
    earliest_rows['severity'] = np.where(earliest_rows['group_p95'] > 0,
                                         earliest_rows['value'] / earliest_rows['group_p95'],
                                         np.nan)
    # Select and rename columns
    out = earliest_rows[['cmdb_id', 'kpi_name', 'group_p95', 'timestamp', 'value', 'severity']].copy()
    out = out.rename(columns={
        'timestamp': 'earliest_gt_p95_ts',
        'value': 'value_at_earliest_gt_p95'
    })
    return out

# 1) metric.csv and log.csv combined
metric_anoms = find_earliest_gt_p95(metric, 'kpi_name', target_services)
log_anoms = find_earliest_gt_p95(log, 'log_name', target_services)

# Combine and sort by severity desc, limit 20
metric_log_combined = pd.concat([metric_anoms, log_anoms], ignore_index=True, sort=False)
metric_log_top20 = metric_log_combined.sort_values('severity', ascending=False).reset_index(drop=True).head(20)

# Counts per service for metric and log (number of distinct groups with any in-window >P95)
metric_counts = metric_anoms.groupby('cmdb_id', as_index=False).agg(metric_gt_groups_count=('kpi_name','nunique'))
log_counts = log_anoms.groupby('cmdb_id', as_index=False).agg(log_gt_groups_count=('kpi_name','nunique'))
svc_summary_metric_log = pd.DataFrame({'cmdb_id': target_services}).merge(metric_counts, on='cmdb_id', how='left') \
                                                            .merge(log_counts, on='cmdb_id', how='left') \
                                                            .fillna(0)
svc_summary_metric_log['metric_gt_groups_count'] = svc_summary_metric_log['metric_gt_groups_count'].astype(int)
svc_summary_metric_log['log_gt_groups_count'] = svc_summary_metric_log['log_gt_groups_count'].astype(int)

metric_log_summary_line = (
    "Metric/log >P95 counts: " +
    "; ".join(f"{row.cmdb_id}: metric={row.metric_gt_groups_count}, log={row.log_gt_groups_count}"
              for row in svc_summary_metric_log.itertuples(index=False))
)

# 2) trace.csv
trace_anoms = find_earliest_gt_p95(trace, 'trace_name', target_services)
# Rename kpi_name to trace_name for output
if not trace_anoms.empty:
    trace_anoms = trace_anoms.rename(columns={'kpi_name': 'trace_name'})
trace_top20 = trace_anoms.sort_values('severity', ascending=False).reset_index(drop=True).head(20)

trace_counts = trace_anoms.groupby('cmdb_id', as_index=False).agg(trace_gt_groups_count=('trace_name','nunique'))
svc_summary_trace = pd.DataFrame({'cmdb_id': target_services}).merge(trace_counts, on='cmdb_id', how='left').fillna(0)
svc_summary_trace['trace_gt_groups_count'] = svc_summary_trace['trace_gt_groups_count'].astype(int)

trace_summary_line = (
    "Trace >P95 counts: " +
    "; ".join(f"{row.cmdb_id}: trace={row.trace_gt_groups_count}"
              for row in svc_summary_trace.itertuples(index=False))
)

# Return compact outputs
metric_log_top20, metric_log_summary_line, trace_top20, trace_summary_line

```
Out[4]:
```
Metric & log anomalies (top by severity, earliest >P95 event shown):

- checkoutservice, log.row_count — p95=100.6, earliest_gt_p95_ts=1705673160, value=114.0, severity=1.1332
- checkoutservice, cpu — p95=0.4751291, earliest_gt_p95_ts=1705673220, value=0.4994899, severity=1.0513
- currencyservice, latency-90 — p95=0.05729414, earliest_gt_p95_ts=1705673400, value=0.05933463, severity=1.0356
- checkoutservice, latency-50 — p95=1.90593, earliest_gt_p95_ts=1705674180, value=1.947965, severity=1.0221
- currencyservice, workload — p95=83.10254, earliest_gt_p95_ts=1705673040, value=84.92290, severity=1.0219
- checkoutservice, latency-90 — p95=4.840979, earliest_gt_p95_ts=1705674240, value=4.92375, severity=1.0171
- (additional groups in table up to 20; entries include currencyservice cpu, currencyservice log.row_count, currencyservice mem, etc.)

Short summary (metric/log): checkoutservice — metric groups with >P95 = 5, log groups with >P95 = 1; currencyservice — metric groups with >P95 = 5, log groups with >P95 = 1.

Trace anomalies (top by severity, earliest >P95 event shown):

- checkoutservice, trace.from_root.duration_p95 — p95=0.919963, earliest_gt_p95_ts=1705673760, value=1.270512, severity=1.3810
- checkoutservice, trace.to_paymentservice.duration_mean — p95=0.000324, earliest_gt_p95_ts=1705673460, value=0.000436, severity=1.3450
- checkoutservice, trace.to_checkoutservice.duration_p95 — p95=1.188384, earliest_gt_p95_ts=1705674300, value=1.455685, severity=1.2249
- checkoutservice, trace.from_checkoutservice.duration_p95 — p95=1.188384, earliest_gt_p95_ts=1705674300, value=1.455685, severity=1.2249
- checkoutservice, trace.to_emailservice.row_count — p95=33.0, earliest_gt_p95_ts=1705673160, value=38.0, severity=1.1515
- checkoutservice, trace.to_paymentservice.row_count — p95=33.0, earliest_gt_p95_ts=1705673160, value=38.0, severity=1.1515
- (additional trace groups up to 20; entries include several checkoutservice row_count and currencyservice trace groups)

Short summary (trace): checkoutservice — trace groups with >P95 = 24; currencyservice — trace groups with >P95 = 8.

Overall quick takeaways:
- The strongest single trace severity appears in checkoutservice (trace.from_root.duration_p95, severity ≈ 1.38 at ts 1705673760).
- For metric/log signals, the largest early anomalies include checkoutservice log.row_count (ts 1705673160) and checkoutservice CPU (ts 1705673220).
- Both checkoutservice and currencyservice have multiple KPI groups exceeding their global P95 during the incident window; counts per file are given above for prioritization.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id       kpi_name     group_p95  earliest_gt_p95_ts  value_at_earliest_gt_p95  severity
0   checkoutservice  log.row_count  1.006000e+02          1705673160              1.140000e+02  1.133201
1   checkoutservice            cpu  4.751291e-01          1705673220              4.994899e-01  1.051272
2   currencyservice     latency-90  5.729414e-02          1705673400              5.933463e-02  1.035614
3   checkoutservice     latency-50  1.905930e+00          1705674180              1.947965e+00  1.022054
4   currencyservice       workload  8.310254e+01          1705673040              8.492290e+01  1.021905
5   checkoutservice     latency-90  4.840979e+00          1705674240              4.923750e+00  1.017098
6   currencyservice            cpu  3.414572e+01          1705673220              3.446769e+01  1.009429
7   checkoutservice       workload  1.008370e+00          1705673160              1.015433e+00  1.007005
8   currencyservice  log.row_count  2.489000e+03          1705673160              2.501000e+03  1.004821
9   currencyservice            mem  4.549032e+07          1705673160              4.569347e+07  1.004466
10  currencyservice     latency-50  4.290613e-03          1705673400              4.305550e-03  1.003481
11  checkoutservice            mem  1.230123e+07          1705673880              1.230452e+07  1.000267, 'Metric/log >P95 counts: checkoutservice: metric=5, log=1; currencyservice: metric=5, log=1',             cmdb_id                                   trace_name   group_p95  earliest_gt_p95_ts  value_at_earliest_gt_p95  severity
0   checkoutservice                 trace.from_root.duration_p95    0.919963          1705673760                  1.270512  1.381046
1   checkoutservice        trace.to_paymentservice.duration_mean    0.000324          1705673460                  0.000436  1.345030
2   checkoutservice        trace.to_checkoutservice.duration_p95    1.188384          1705674300                  1.455685  1.224928
3   checkoutservice      trace.from_checkoutservice.duration_p95    1.188384          1705674300                  1.455685  1.224928
4   checkoutservice              trace.to_emailservice.row_count   33.000000          1705673160                 38.000000  1.151515
5   checkoutservice            trace.to_paymentservice.row_count   33.000000          1705673160                 38.000000  1.151515
6   currencyservice     trace.from_frontendservice.duration_mean    0.000579          1705673040                  0.000653  1.127708
7   checkoutservice         trace.to_paymentservice.duration_p95    0.000767          1705673460                  0.000864  1.126082
8   checkoutservice           trace.to_checkoutservice.row_count  295.000000          1705673160                330.000000  1.118644
9   checkoutservice         trace.from_checkoutservice.row_count  295.000000          1705673160                330.000000  1.118644
10  checkoutservice                trace.from_root.duration_mean    0.224216          1705673760                  0.246154  1.097843
11  currencyservice         trace.from_checkoutservice.row_count  127.600000          1705673160                140.000000  1.097179
12  checkoutservice           trace.to_currencyservice.row_count  127.600000          1705673160                140.000000  1.097179
13  checkoutservice                    trace.from_root.row_count  150.000000          1705673160                164.000000  1.093333
14  checkoutservice         trace.from_frontendservice.row_count   34.800000          1705673160                 38.000000  1.091954
15  checkoutservice  trace.to_productcatalogservice.duration_p95    0.000034          1705673100                  0.000037  1.088804
16  checkoutservice       trace.to_checkoutservice.duration_mean    0.320425          1705674300                  0.348613  1.087971
17  checkoutservice     trace.from_checkoutservice.duration_mean    0.320425          1705674300                  0.348613  1.087971
18  currencyservice                 trace.from_root.duration_p95    0.000361          1705674060                  0.000391  1.084824
19  checkoutservice     trace.to_productcatalogservice.row_count   94.600000          1705673160                102.000000  1.078224, 'Trace >P95 counts: checkoutservice: trace=24; currencyservice: trace=8')```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse existing DataFrames and window variables from kernel: metric, trace, log, win_start, win_end
service = 'currencyservice'

# Ensure columns/types
for df in (metric, trace, log):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['ts_dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

def extract_gt_p95(df, kpi_col, file_label):
    # Restrict to service
    df_sel = df[df['cmdb_id'] == service].copy()
    if df_sel.empty:
        return pd.DataFrame(columns=['file','group_name','group_p95','earliest_gt_p95_ts','value_at_earliest_gt_p95','severity'])
    # compute global p95 per group using full series (no time filtering)
    p95 = (
        df_sel.groupby(kpi_col, as_index=False)
        .agg(group_p95=('value', lambda x: float(x.quantile(0.95)) if x.notna().any() else np.nan))
    )
    merged = df_sel.merge(p95, on=kpi_col, how='left')
    # filter to incident window and to rows > group_p95
    in_win = merged[(merged['ts_dt'] >= win_start) & (merged['ts_dt'] <= win_end)].copy()
    in_win = in_win[in_win['value'] > in_win['group_p95']]
    if in_win.empty:
        return pd.DataFrame(columns=['file','group_name','group_p95','earliest_gt_p95_ts','value_at_earliest_gt_p95','severity'])
    # earliest row per group (by timestamp)
    idx = in_win.groupby(kpi_col)['timestamp'].idxmin()
    rows = in_win.loc[idx].copy()
    # compute severity safely (avoid divide by zero)
    rows['severity'] = np.where(rows['group_p95'] > 0, rows['value'] / rows['group_p95'], np.nan)
    out = rows[[kpi_col, 'group_p95', 'timestamp', 'value', 'severity']].copy()
    out = out.rename(columns={
        kpi_col: 'group_name',
        'timestamp': 'earliest_gt_p95_ts',
        'value': 'value_at_earliest_gt_p95'
    })
    out.insert(0, 'file', file_label)
    # keep column order
    out = out[['file','group_name','group_p95','earliest_gt_p95_ts','value_at_earliest_gt_p95','severity']]
    return out

# Extract from each file
metric_events = extract_gt_p95(metric, 'kpi_name', 'metric')
trace_events = extract_gt_p95(trace, 'trace_name', 'trace')
log_events = extract_gt_p95(log, 'log_name', 'log')

# Combine and rank by severity
all_events = pd.concat([metric_events, trace_events, log_events], ignore_index=True, sort=False)
# If severity NaN (due to p95==0), keep but they will sort last; drop rows with NaN severity to focus on numeric ranking
all_events_numeric = all_events[all_events['severity'].notna()].copy()
top10_anoms = all_events_numeric.sort_values('severity', ascending=False).head(10).reset_index(drop=True)

# total number of distinct groups for currencyservice that had any in-window >P95 (count unique file+group_name)
total_groups_count = all_events[['file','group_name']].drop_duplicates().shape[0]

# Display compact results
top10_anoms, total_groups_count

```
Out[5]:
```
Top 10 in-window (>P95) anomaly events for currencyservice (ranked by severity):

1) trace — trace.from_frontendservice.duration_mean: p95=0.000579, earliest_ts=1705673040, value=0.000653, severity=1.127708  
2) trace — trace.from_checkoutservice.row_count: p95=127.6, earliest_ts=1705673160, value=140.0, severity=1.097179  
3) trace — trace.from_root.duration_p95: p95=0.000361, earliest_ts=1705674060, value=0.000391, severity=1.084824  
4) trace — trace.from_root.duration_mean: p95=0.000103, earliest_ts=1705674060, value=0.000109, severity=1.050425  
5) trace — trace.from_checkoutservice.duration_mean: p95=0.001619, earliest_ts=1705673160, value=0.001683, severity=1.039264  
6) metric — latency-90: p95=0.057294, earliest_ts=1705673400, value=0.059335, severity=1.035614  
7) metric — workload: p95=83.102543, earliest_ts=1705673040, value=84.922897, severity=1.021905  
8) trace — trace.from_frontendservice.duration_p95: p95=0.000263, earliest_ts=1705673400, value=0.000267, severity=1.013398  
9) metric — cpu: p95=34.145721, earliest_ts=1705673220, value=34.467694, severity=1.009429  
10) log — log.row_count: p95=2489.0, earliest_ts=1705673160, value=2501.0, severity=1.004821

Total distinct currencyservice groups with any in-window >P95: 14.

The original code execution output of IPython Kernel is also provided below for reference:

(     file                                group_name    group_p95  earliest_gt_p95_ts  value_at_earliest_gt_p95  severity
0   trace  trace.from_frontendservice.duration_mean     0.000579          1705673040                  0.000653  1.127708
1   trace      trace.from_checkoutservice.row_count   127.600000          1705673160                140.000000  1.097179
2   trace              trace.from_root.duration_p95     0.000361          1705674060                  0.000391  1.084824
3   trace             trace.from_root.duration_mean     0.000103          1705674060                  0.000109  1.050425
4   trace  trace.from_checkoutservice.duration_mean     0.001619          1705673160                  0.001683  1.039264
5  metric                                latency-90     0.057294          1705673400                  0.059335  1.035614
6  metric                                  workload    83.102543          1705673040                 84.922897  1.021905
7   trace   trace.from_frontendservice.duration_p95     0.000263          1705673400                  0.000267  1.013398
8  metric                                       cpu    34.145721          1705673220                 34.467694  1.009429
9     log                             log.row_count  2489.000000          1705673160               2501.000000  1.004821, 14)```
```